# K5 × a clip gate fitted on `nexar_train` (`NG`), beside §27's MSAD gate (`MS`)

Rule: `core/docs/final/KIM_RUN.md` §20, written before any number. **PRELIMINARY (n = 1)**. Nothing in K5 is trained: only a 513-parameter logistic gate on mean raw CLIP. Macro is unchanged by construction (`c(v) · minmax_v(s)`; the report raises if it moves). `nexar_test` is never touched. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, deps, preflight (K5, census, caches, §27 gate, K5's saved TAD / DAD / MSAD curves) | no | 2-5 min |
| 4 | pytest on the touched code — **raises if red** | no | 1-2 min |
| 5 | fit `NG` on the 900 `nexar_train` videos (R4: 451 / 449) | no | 10-25 min (staging 900 s1 files) |
| 6 | score `NG` and `MS` on `nexar_val`, TAD, DAD, MSAD (R2 in the report) | no | 5-10 min |
| 7 | K5 re-read on `nexar_val` with `--save-scores` + **R1** (micro, crop endpoint == §13) | yes | 10-20 min |
| 8 | `core.tools.k5_clip_gate` -> **paste `REPORTS/k5_clip_gate.md` to Claude** | no | 3-6 min |

A re-run skips what is on Drive. Thesis and Thesis-V2 are only read; everything new goes to `Thesis-Final/outputs/kim_gate`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: cache/clip/{TAD_ncc,MSAD_ncc}, data/TAD
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar caches, census, A2 stats, Nexar_window, §27 gate
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5 and its reads, DAD cache; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert 'def strata_read' in (REPO / 'core/tools/k5_clip_gate.py').read_text(), 'old k5_clip_gate -- re-upload kat-vad/'
assert '--save-scores' in (REPO / 'core/tools/nexar_eval.py').read_text(), 'nexar_eval has no --save-scores -- re-upload kat-vad/'
assert '## 20. A clip gate fitted on `nexar_train`' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §20 -- re-upload'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5, SEED = constants.KIM_MAIN_ARM, constants.KIM_SEED            # K5, 2024
ENC      = constants.V5_ENCODER                                  # V2-S
DATASET  = constants.NEXAR_DATASET
NEXAR_SPLIT = constants.V2_SPLIT_NEXAR_VAL                       # nexar_test is never read here
V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'

# Nexar
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET          # raw CLIP, every native frame
NEXAR_U  = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                       # K5's input statistics
WINDOW   = Path(DRIVE_V2) / 'data' / f'{DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'   # K5's defs
# K5
KIM      = constants.OUTPUT_ROOT / constants.KIM_OUTPUT_DIRNAME
K5_CKPT  = KIM / K5 / f's{SEED}' / 'checkpoint_last.pt'
K5_REF   = constants.OUTPUT_ROOT / constants.KIM_T2_OUTPUT_DIRNAME / 'eval' / 'nexar' / K5   # §13's read (R1)
K5_RUNS  = {'TAD': KIM / 'eval' / 'tad' / K5 / f's{SEED}',
            'DAD': KIM / 'eval' / constants.DAD_OUTPUT_SET / K5 / f's{SEED}',
            'MSAD': constants.OUTPUT_ROOT / constants.MSAD_ZS_OUTPUT_DIRNAME / 'eval' / K5 / f's{SEED}'}
K5_COUNTS = {'TAD': constants.V2_TADZ_VIDEOS, 'DAD': constants.DAD_TEST_VIDEOS, 'MSAD': constants.MSAD_ZS_TEST_VIDEOS}
# Raw CLIP of the zero-shot sets (the gate's input; same _ncc extractor, C2)
CLIP     = {'TAD': Path(DRIVE) / 'cache' / 'clip' / 'TAD_ncc',
            'DAD': constants.CACHE_ROOT / 'clip' / f'{constants.DAD_DATASET}_s{constants.DAD_STRIDE}_ncc',
            'MSAD': Path(DRIVE) / 'cache' / 'clip' / 'MSAD_ncc'}
# §27's gate (MSAD-train) and its recorded scores (re-score check)
MS_DIR   = V2_OUT / 'v2' / 'v2_tadg' / 'gate'
# Outputs
ROOT     = constants.OUTPUT_ROOT / constants.KIM_GATE_OUTPUT_DIRNAME
GATES    = ROOT / 'gates'
EVAL     = ROOT / 'eval' / 'nexar' / K5
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/k5_gate')
LOCAL.mkdir(parents=True, exist_ok=True)

INPUTS = (('Nexar census', CENSUS), ('Nexar CLIP s1', NEXAR_X), ('Nexar V2-S s1', NEXAR_U),
          ('Nexar A2 stats', A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME), ('Nexar_window defs', WINDOW),
          ('K5 checkpoint', K5_CKPT), ('K5 nexar_val read (§13)', K5_REF / 'nexar_readout.json'),
          ('§27 gate', MS_DIR / 'gate.npz'), ('§27 TAD scores', MS_DIR / 'c_test.json'),
          *[(f'K5 {s} curves', d / 'scores') for s, d in K5_RUNS.items()],
          *[(f'{s} raw CLIP', d) for s, d in CLIP.items()])
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | outputs -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 7 needs one'
df -h /content | tail -1

In [ ]:
from core.data.v2_splits import load_split            # noqa: E402
from core.tools.nexar_eval import ITEM_AUCS_JSON, READOUT_JSON  # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json/npz manifests if given) to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


# Preflight on small files only (C39)
census = json.loads(CENSUS.read_text())['videos']
TRAIN_IDS = sorted(load_split(constants.V2_SPLIT_NEXAR_TRAIN))
NEXAR_IDS = sorted(load_split(NEXAR_SPLIT))
counts = (sum(census[v]['label'] for v in TRAIN_IDS), sum(1 - census[v]['label'] for v in TRAIN_IDS))
assert counts == constants.KIM_GATE_NEXAR_TRAIN_COUNTS, f'R4: nexar_train is {counts}'
assert not set(TRAIN_IDS) & set(NEXAR_IDS), 'nexar_train and nexar_val overlap'
SET_IDS = {s: sorted(p.stem for p in (d / 'scores').glob('*.npz')) for s, d in K5_RUNS.items()}
for s, ids in SET_IDS.items():
    assert len(ids) == K5_COUNTS[s], f'K5 {s} curves: {len(ids)} files, expected {K5_COUNTS[s]}'
    miss = [v for v in ids if not (CLIP[s] / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} {s} videos have no raw CLIP in {CLIP[s]}: {miss[:5]}'
miss = [v for v in TRAIN_IDS + NEXAR_IDS if not (NEXAR_X / f'{v}.npy').exists()]
assert not miss, f'{len(miss)} Nexar videos have no CLIP s1: {miss[:5]}'
META = sorted({k for v in NEXAR_IDS for k in census[v] if k.startswith('meta_')})
print(f'preflight OK: nexar_train {len(TRAIN_IDS)} {counts} | {NEXAR_SPLIT} {len(NEXAR_IDS)} | '
      + ', '.join(f'{s} {len(i)}' for s, i in SET_IDS.items()) + f' | census meta keys {META}')

## 4. Tests on the touched code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_k5_clip_gate.py',
     'core/tests/test_tad_clip_gate.py', 'core/tests/test_nexar_pipeline.py'])

## 5. Fit `NG` on every `nexar_train` video (§20 NX1; R4 = `--expect-counts 451 449`)

`tad_clip_gate fit` unchanged: standardized mean raw CLIP, class-balanced logistic, `C` = 1.0. No `nexar_val` video is fitted on.

In [ ]:
NG = GATES / 'NG'
if not (NG / 'gate.npz').exists():
    local = LOCAL / 'NG'
    local.mkdir(parents=True, exist_ok=True)
    (local / 'labels_train.json').write_text(json.dumps({v: int(census[v]['label']) for v in TRAIN_IDS}))
    l_x = stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', TRAIN_IDS + NEXAR_IDS)
    run([sys.executable, '-m', 'core.tools.tad_clip_gate', 'fit', '--clip-dir', l_x,
         '--labels', local / 'labels_train.json', '--out', local / 'gate.npz',
         '--expect-counts', *constants.KIM_GATE_NEXAR_TRAIN_COUNTS], log=LOCAL / 'fit.log')
    publish(local, NG)
    flush_drive()
fit = json.loads((NG / 'gate.json').read_text())
assert tuple(fit['abnormal_normal']) == constants.KIM_GATE_NEXAR_TRAIN_COUNTS and fit['videos'] == len(TRAIN_IDS), fit
print(f"NG: fitted on {fit['videos']} videos {fit['abnormal_normal']} | train clip AUC {fit['train_clip_auc']:.4f} (in-sample)")

## 6. Score `NG` and `MS` on `nexar_val`, TAD, DAD, MSAD (§20 NX3)

Check: `MS` re-scored on TAD must equal §27's recorded `c_test.json` (same gate, same features).

In [ ]:
SCORE_SETS = {'nexar_val': (NEXAR_X, NEXAR_IDS), **{s: (CLIP[s], SET_IDS[s]) for s in CLIP}}
GATE_NPZ = {'NG': NG / 'gate.npz', 'MS': MS_DIR / 'gate.npz'}
for s, (clip_dir, ids) in SCORE_SETS.items():
    outs = {g: GATES / g / f'c_{s}.json' for g in GATE_NPZ}
    if all(o.exists() for o in outs.values()):
        continue
    l_clip = stage(clip_dir, LOCAL / 'stage' / ('nexar_x' if s == 'nexar_val' else f'{s}_clip'), ids)
    ids_file = LOCAL / f'ids_{s}.txt'
    ids_file.write_text('\n'.join(ids) + '\n')
    for g, npz in GATE_NPZ.items():
        local = LOCAL / 'scores' / g / f'c_{s}.json'
        run([sys.executable, '-m', 'core.tools.tad_clip_gate', 'score', '--gate', npz, '--clip-dir', l_clip,
             '--ids-file', ids_file, '--out', local])
        publish(local, outs[g])
flush_drive()
old = json.loads((MS_DIR / 'c_test.json').read_text())
new = json.loads((GATES / 'MS' / 'c_TAD.json').read_text())
gap = max(abs(old[v]['c'] - new[v]['c']) for v in new)
assert set(old) == set(new) and gap < 1e-9, f'MS re-score differs from §27 by {gap:.2e}'
print(f'MS re-score on TAD == §27 c_test.json (max gap {gap:.1e})')

## 7. K5 on `nexar_val`, whole videos saved (`--save-scores`) + **R1**

R1: whole micro and crop endpoint equal §13's recorded read within `KIM_GATE_REPRO_TOL` (1e-4), else stop.

In [ ]:
import torch  # noqa: E402

if not (EVAL / 'scores').is_dir() or len(list((EVAL / 'scores').glob('*.npz'))) != len(NEXAR_IDS):
    assert torch.cuda.is_available(), 'no GPU -- this step scores K5'
    l_x = stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', NEXAR_IDS)
    l_u = stage(NEXAR_U, LOCAL / 'stage' / 'nexar_u', NEXAR_IDS)
    l_stats = stage(A2_NEXAR, LOCAL / 'stage' / 'nexar_stats', [])     # manifest + statistics only
    l_defs = LOCAL / 'nexar_defs'
    l_defs.mkdir(parents=True, exist_ok=True)
    for f in WINDOW.glob('*.json'):
        shutil.copy2(f, l_defs / f.name)
    local = LOCAL / 'eval' / 'nexar'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.nexar_eval', '--run', f's{SEED}', K5_CKPT, '--stats-dir', l_stats,
         '--video-s1-dir', l_u, '--census', CENSUS, '--clip-s1-dir', l_x, '--data-dir', l_defs,
         '--split', NEXAR_SPLIT, '--save-scores', '--out-dir', local], log=LOCAL / 'nexar.log')
    publish(local, EVAL)
    flush_drive()
got = json.loads((EVAL / READOUT_JSON).read_text())
ref = json.loads((K5_REF / READOUT_JSON).read_text())
assert got['split'] == NEXAR_SPLIT and got['videos'] == len(NEXAR_IDS), (got['split'], got['videos'])
for key, read in (('whole micro', lambda r: r['whole']['micro']),
                  ('crop endpoint', lambda r: r['crops']['mean_over_placements']['macro']['mean'])):
    gap = read(got) - read(ref)
    print(f'R1 {key}: re-read {read(got):.6f} vs §13 {read(ref):.6f} (gap {gap:+.1e})')
    assert abs(gap) <= constants.KIM_GATE_REPRO_TOL, f'R1 FAILED on {key} -- stop and paste this to Claude'

## 8. The table -> **paste `REPORTS/k5_clip_gate.md` to Claude**

In [ ]:
strata = {v: {k[len('meta_'):]: census[v][k] for k in META if k in census[v]} for v in NEXAR_IDS}
(LOCAL / 'strata.json').write_text(json.dumps(strata))
RUN_DIRS = {'nexar_val': EVAL, 'TAD': K5_RUNS['TAD'], 'DAD': K5_RUNS['DAD'], 'MSAD': K5_RUNS['MSAD']}
cmd = [sys.executable, '-m', 'core.tools.k5_clip_gate']
for s, d in RUN_DIRS.items():
    cmd += ['--set', s, d, *[f'{g}={GATES / g / f"c_{s}.json"}' for g in ('NG', 'MS')]]
cmd += ['--in-domain', 'MS:MSAD', '--strata', 'nexar_val', LOCAL / 'strata.json',
        '--out-dir', LOCAL / 'report']
run(cmd)
publish(LOCAL / 'report', REPORTS)
publish(LOCAL / 'strata.json', REPORTS / 'strata_nexar_val.json')
print((REPORTS / f'{constants.K5G_REPORT_STEM}.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/k5_clip_gate.md to Claude')